# P1 Chunking benchmark (Primary): SciFact and FiQA

Compares **fixed-length, recursive and semantic chunking** at two chunk sizes (128 and 256 tokens)
against a no-chunking baseline (★), with the open `BAAI/bge-small-en-v1.5` embedder. BM25 on whole documents
is added as a lexical reference. No LLM is used (retrieval-only project).

Reports Recall@5, Recall@10, MRR@10 and NDCG@10 as mean ± std over 3 seeds (T1), number of chunks and
indexing cost (T2), an NDCG@10 grouped bar chart (F1) and Recall@K curves for K = 1..20 (F2).
Each seed samples 200 test queries and a 3,000-document corpus (all gold docs + random distractors);
change `N_QUERIES` / `CORPUS_SIZE` in `p1/config.py` (cell 2) if your problem statement gives another subset size.

**Before you start:** Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.
No uploads needed. Results are written to `results/`, zipped and downloaded; the last cell starts the
**web frontend** (a public `gradio.live` link). Expected run time: roughly 10-20 minutes on a T4 (not yet timed).

## 1. Check the GPU and install packages

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q sentence-transformers datasets rank_bm25 gradio tabulate matplotlib pandas

## 2. Write the project code (p1 package)

In [ ]:
import os
os.makedirs('/content/p1proj/p1', exist_ok=True)
%cd /content/p1proj
open('p1/__init__.py','w').write('')

In [ ]:
%%writefile p1/config.py
"""Configuration for P1 (Primary): chunking benchmark on SciFact and FiQA.

Environment overrides: P1_ROOT (output root), P1_SMOKE=1 (tiny run for testing only; never report it).
"""
import os
from pathlib import Path

ROOT = Path(os.environ.get("P1_ROOT", Path(__file__).resolve().parents[1]))
RESULTS = ROOT / "results"

SMOKE = os.environ.get("P1_SMOKE") == "1"

# ---- model (open, small) -------------------------------------------------
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # BGE retrieval instruction
MAX_SEQ_LEN = 512
EMBED_BATCH = 128
# No LLM: P1 is a retrieval-only benchmark (0 LLM tokens).

# ---- experiment grid -----------------------------------------------------
DATASETS = ["scifact", "fiqa"]
STRATEGIES = ["fixed", "recursive", "semantic"]
CHUNK_SIZES = [128, 256]            # max tokens per chunk (BGE tokenizer tokens)
CHUNK_OVERLAP = 0                   # same (zero) overlap for every strategy
SEMANTIC_PERCENTILE = 90            # cut where adjacent-sentence distance > this per-doc percentile
BASELINE = "fulldoc"                # no chunking: whole document (model truncates at MAX_SEQ_LEN)

# ---- data subset ---------------------------------------------------------
SEEDS = [13, 42, 2024]              # each seed = a different query sample + distractor sample
N_QUERIES = 200                     # test queries per seed
CORPUS_SIZE = 3000                  # docs per seed: all gold docs of those queries + random distractors

# ---- evaluation ----------------------------------------------------------
K_MAX = 20                          # Recall@K curve, K = 1..20

if SMOKE:                           # never use for reported numbers
    SEEDS = [13, 42]
    N_QUERIES = 30
    CORPUS_SIZE = 300

In [ ]:
%%writefile p1/embed.py
"""BGE-small embedder + tokenizer helpers shared by the chunkers, the runner and the frontend."""
import platform

import numpy as np

from . import config as C


class Embedder:
    def __init__(self, model_name=C.EMBED_MODEL):
        import torch
        import transformers
        from sentence_transformers import SentenceTransformer
        transformers.logging.set_verbosity_error()
        self.torch = torch
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(model_name, device=self.device)
        self.model.max_seq_length = C.MAX_SEQ_LEN
        if self.device == "cuda":
            self.model.half()                       # fp16 on GPU for speed
        self.tokenizer = self.model.tokenizer
        self.dim = self.model.get_sentence_embedding_dimension()
        self.hardware = torch.cuda.get_device_name(0) if self.device == "cuda" else (platform.processor() or "CPU")
        self.embed(["warm up the model"] * 8)       # keep CUDA start-up out of the timings

    def n_tokens(self, texts):
        """Content tokens (no [CLS]/[SEP]) with the embedder's own tokenizer."""
        if isinstance(texts, str):
            texts = [texts]
        if not texts:
            return []
        return [len(x) for x in self.tokenizer(list(texts), add_special_tokens=False, truncation=False)["input_ids"]]

    def embed(self, texts, is_query=False):
        texts = [C.QUERY_PREFIX + t for t in texts] if is_query else list(texts)
        if not texts:
            return np.zeros((0, self.dim), dtype=np.float32)
        E = self.model.encode(texts, batch_size=C.EMBED_BATCH, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False)
        if self.device == "cuda":
            self.torch.cuda.synchronize()
        return E.astype(np.float32)


def billed(lens):
    """Tokens actually seen by the encoder (truncated at MAX_SEQ_LEN, incl. 2 special tokens)."""
    return int(sum(min(l, C.MAX_SEQ_LEN - 2) + 2 for l in lens))

In [ ]:
%%writefile p1/data.py
"""Load SciFact / FiQA (BEIR test split) and build seeded subsets (never the full datasets)."""
import io
import json
import os
import zipfile
from collections import defaultdict

import numpy as np

from . import config as C

BEIR_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip"


def _load_hf(name):
    from datasets import load_dataset
    c = load_dataset(f"BeIR/{name}", "corpus", split="corpus")
    q = load_dataset(f"BeIR/{name}", "queries", split="queries")
    r = load_dataset(f"BeIR/{name}-qrels", split="test")
    corpus = {str(x["_id"]): {"title": x.get("title") or "", "text": x.get("text") or ""} for x in c}
    queries = {str(x["_id"]): x["text"] for x in q}
    qrels = defaultdict(dict)
    for x in r:
        if int(x["score"]) > 0:
            qrels[str(x["query-id"])][str(x["corpus-id"])] = int(x["score"])
    return corpus, queries, dict(qrels)


def _load_zip(name, root="beir_data"):
    import requests
    path = os.path.join(root, name)
    if not os.path.exists(path):
        os.makedirs(root, exist_ok=True)
        data = requests.get(BEIR_URL.format(name), timeout=300).content
        zipfile.ZipFile(io.BytesIO(data)).extractall(root)
    corpus, queries, qrels = {}, {}, defaultdict(dict)
    with open(os.path.join(path, "corpus.jsonl")) as f:
        for line in f:
            x = json.loads(line)
            corpus[str(x["_id"])] = {"title": x.get("title") or "", "text": x.get("text") or ""}
    with open(os.path.join(path, "queries.jsonl")) as f:
        for line in f:
            x = json.loads(line)
            queries[str(x["_id"])] = x["text"]
    with open(os.path.join(path, "qrels", "test.tsv")) as f:
        next(f)
        for line in f:
            qid, did, s = line.rstrip("\n").split("\t")
            if int(s) > 0:
                qrels[qid][did] = int(s)
    return corpus, queries, dict(qrels)


def load_all(datasets=None):
    """Returns {name: (corpus, queries, qrels)}; Hugging Face first, official BEIR zip as fallback."""
    out = {}
    for name in datasets or C.DATASETS:
        try:
            out[name] = _load_hf(name)
            src = "Hugging Face"
        except Exception as e:
            print(f"[{name}] HF load failed ({type(e).__name__}); using BEIR zip")
            out[name] = _load_zip(name)
            src = "BEIR zip"
        c, q, r = out[name]
        print(f"{name:8s} ({src}): corpus={len(c):,}  queries={len(q):,}  test queries with qrels={len(r):,}")
    return out


def doc_text(d):
    t, x = d["title"].strip(), d["text"].strip()
    return f"{t}\n\n{x}" if t else x


def make_subset(data, name, seed):
    corpus, queries, qrels = data[name]
    rng = np.random.default_rng(seed)
    eligible = sorted(q for q, rel in qrels.items() if q in queries and rel and all(d in corpus for d in rel))
    qids = sorted(rng.choice(eligible, min(C.N_QUERIES, len(eligible)), replace=False).tolist())
    gold = set(d for q in qids for d in qrels[q])
    pool = sorted(set(corpus) - gold)
    n_extra = max(0, min(C.CORPUS_SIZE - len(gold), len(pool)))
    extra = rng.choice(pool, n_extra, replace=False).tolist() if n_extra else []
    doc_ids = sorted(gold | set(extra))
    if len(gold) > C.CORPUS_SIZE:
        print(f"  note: {len(gold)} gold docs > CORPUS_SIZE; corpus holds gold docs only")
    return dict(name=name, seed=seed, qids=qids, queries=[queries[q] for q in qids],
                qrels={q: qrels[q] for q in qids}, doc_ids=doc_ids,
                docs=[doc_text(corpus[d]) for d in doc_ids])

In [ ]:
%%writefile p1/chunkers.py
"""The three chunking strategies (+ "fulldoc" = no-chunking baseline).

All sizes are in the embedder's (BGE WordPiece) tokens. Overlap = C.CHUNK_OVERLAP for every strategy.
  fixed     - consecutive windows of exactly `size` tokens; cuts ignore sentence boundaries
  recursive - split on paragraph -> line -> sentence -> word, greedily pack pieces up to `size`
  semantic  - embed sentences, start a new chunk where adjacent-sentence cosine distance exceeds
              the document's SEMANTIC_PERCENTILE-th percentile; `size` is a hard cap
"""
import re
import time

import numpy as np

from . import config as C
from .embed import billed

SEPARATORS = ["\n\n", "\n", ". ", " "]
_SENT_RE = re.compile(r'(?<=[.!?])\s+(?=[A-Z0-9"\'(\[])|\n+')


def split_sentences(text):
    return [s.strip() for s in _SENT_RE.split(text) if s and s.strip()]


class Chunker:
    """`emb` must provide .tokenizer (fast HF tokenizer), .n_tokens(texts) and .embed(texts)."""

    def __init__(self, emb):
        self.emb = emb
        self.tok = emb.tokenizer

    # ---- fixed-length ------------------------------------------------------
    def fixed(self, text, size):
        enc = self.tok(text, add_special_tokens=False, return_offsets_mapping=True, truncation=False)
        offs = enc["offset_mapping"]
        if not offs:
            return [text.strip()] if text.strip() else []
        step = max(1, size - C.CHUNK_OVERLAP)
        out = []
        for i in range(0, len(offs), step):
            s, e = offs[i][0], offs[min(i + size, len(offs)) - 1][1]
            out.append(text[s:e].strip())
            if i + size >= len(offs):
                break
        return [c for c in out if c]

    # ---- recursive ---------------------------------------------------------
    def recursive(self, text, size, seps=SEPARATORS):
        text = text.strip()
        if not text:
            return []
        if self.emb.n_tokens(text)[0] <= size:
            return [text]
        for i, sep in enumerate(seps):
            if sep in text:
                break
        else:
            return self.fixed(text, size)
        parts = text.split(sep)
        pieces = [p + sep for p in parts[:-1]] + [parts[-1]]
        pieces = [p for p in pieces if p.strip()]
        lens = self.emb.n_tokens(pieces)
        chunks, cur, cur_len = [], "", 0
        for p, l in zip(pieces, lens):
            if l > size:
                if cur.strip():
                    chunks.append(cur.strip())
                cur, cur_len = "", 0
                chunks.extend(self.recursive(p, size, seps[i + 1:]))
            elif cur_len + l <= size:
                cur += p
                cur_len += l
            else:
                chunks.append(cur.strip())
                cur, cur_len = p, l
        if cur.strip():
            chunks.append(cur.strip())
        out = []
        for c, l in zip(chunks, self.emb.n_tokens(chunks)):     # hard cap (token counts only ~additive)
            out.extend(self.fixed(c, size) if l > size else [c])
        return out

    # ---- semantic ----------------------------------------------------------
    def prepare_sentences(self, docs):
        """Sentence split + sentence embeddings, shared by both chunk sizes. Its full time and tokens
        are charged to EACH semantic configuration in T2."""
        t0 = time.perf_counter()
        per_doc = [split_sentences(d) for d in docs]
        flat = [s for ss in per_doc for s in ss]
        lens = self.emb.n_tokens(flat)
        E = self.emb.embed(flat)
        sents, embs, lns, k = [], [], [], 0
        for ss in per_doc:
            sents.append(ss)
            embs.append(E[k:k + len(ss)])
            lns.append(lens[k:k + len(ss)])
            k += len(ss)
        return dict(sents=sents, embs=embs, lens=lns, time=time.perf_counter() - t0,
                    tokens=billed(lens), n=len(flat))

    def semantic(self, sents, embs, lens, size, pct=C.SEMANTIC_PERCENTILE):
        if not sents:
            return []
        groups = [[0]]
        if len(sents) > 1:
            dist = 1.0 - np.sum(embs[:-1] * embs[1:], axis=1)
            thr = np.percentile(dist, pct)
            for i in range(1, len(sents)):
                if dist[i - 1] > thr:
                    groups.append([i])
                else:
                    groups[-1].append(i)
        chunks = []
        for g in groups:
            cur, cur_len = [], 0
            for i in g:
                if lens[i] > size:                      # single over-long sentence -> token windows
                    if cur:
                        chunks.append(" ".join(sents[j] for j in cur))
                    cur, cur_len = [], 0
                    chunks.extend(self.fixed(sents[i], size))
                    continue
                if cur and cur_len + lens[i] > size:
                    chunks.append(" ".join(sents[j] for j in cur))
                    cur, cur_len = [], 0
                cur.append(i)
                cur_len += lens[i]
            if cur:
                chunks.append(" ".join(sents[j] for j in cur))
        return chunks

    # ---- corpus ------------------------------------------------------------
    def chunk_corpus(self, docs, strategy, size, sent_cache=None):
        """Returns (chunks, owner) where owner[i] = index of the doc chunk i came from (sorted)."""
        if strategy == "semantic" and sent_cache is None:
            sent_cache = self.prepare_sentences(docs)
        chunks, owner = [], []
        for di, d in enumerate(docs):
            if strategy == "fixed":
                cs = self.fixed(d, size)
            elif strategy == "recursive":
                cs = self.recursive(d, size)
            elif strategy == "semantic":
                cs = self.semantic(sent_cache["sents"][di], sent_cache["embs"][di], sent_cache["lens"][di], size)
            elif strategy == "fulldoc":
                cs = [d]
            else:
                raise ValueError(strategy)
            if not cs:
                cs = [d if d.strip() else "(empty)"]  # every doc keeps >= 1 chunk
            chunks.extend(cs)
            owner.extend([di] * len(cs))
        return chunks, np.asarray(owner)

In [ ]:
%%writefile p1/retrieval.py
"""Dense retrieval with MaxP (doc score = best chunk score) and the BM25 lexical reference."""
import re

import numpy as np

from . import config as C

_tok_re = re.compile(r"\w+")


def bm25_tok(t):
    return _tok_re.findall(t.lower())


def chunk_starts(owner, n_docs):
    return np.searchsorted(owner, np.arange(n_docs))      # owner is sorted, every doc has >= 1 chunk


def doc_scores_maxp(Q, C_emb, owner, n_docs):
    """(n_queries, n_docs) matrix of best-chunk cosine scores."""
    return np.maximum.reduceat(Q @ C_emb.T, chunk_starts(owner, n_docs), axis=1)


def top_k(D, k=C.K_MAX):
    return np.argsort(-D, axis=1, kind="stable")[:, :k]

In [ ]:
%%writefile p1/metrics.py
"""Recall@1..K, MRR@10, NDCG@10 (linear graded gain).   Self-test:  python -m p1.metrics"""
import math
from collections import defaultdict

import numpy as np

from . import config as C


def evaluate(ranked_ids, qids, qrels, k_max=C.K_MAX):
    """ranked_ids[i] = ranked doc ids for qids[i]; qrels[q] = {doc_id: grade>0}. Returns mean metrics."""
    per = defaultdict(list)
    for qi, q in enumerate(qids):
        rel, ranked = qrels[q], ranked_ids[qi]
        hits = [1 if d in rel else 0 for d in ranked]
        for k in range(1, k_max + 1):
            per[f"R@{k}"].append(sum(hits[:k]) / len(rel))
        per["MRR@10"].append(next((1.0 / (i + 1) for i, h in enumerate(hits[:10]) if h), 0.0))
        dcg = sum(rel.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked[:10]))
        idcg = sum(g / math.log2(i + 2) for i, g in enumerate(sorted(rel.values(), reverse=True)[:10]))
        per["NDCG@10"].append(dcg / idcg if idcg else 0.0)
    return {k: float(np.mean(v)) for k, v in per.items()}


def self_test():
    ranked = [["d1", "d2", "d3"] + [f"x{i}" for i in range(17)]]
    m = evaluate(ranked, ["a"], {"a": {"d2": 1, "d9": 1}})
    assert abs(m["MRR@10"] - 0.5) < 1e-9
    assert abs(m["R@1"] - 0.0) < 1e-9 and abs(m["R@5"] - 0.5) < 1e-9 and abs(m["R@20"] - 0.5) < 1e-9
    assert abs(m["NDCG@10"] - (1 / math.log2(3)) / (1 + 1 / math.log2(3))) < 1e-9
    m = evaluate([["d9", "d2"] + [f"x{i}" for i in range(18)]], ["a"], {"a": {"d2": 1, "d9": 1}})
    assert abs(m["NDCG@10"] - 1.0) < 1e-9 and abs(m["MRR@10"] - 1.0) < 1e-9
    print("metric self-test passed")


if __name__ == "__main__":
    self_test()

In [ ]:
%%writefile p1/run_experiment.py
"""Run the whole chunking benchmark:  python -m p1.run_experiment

For every (dataset, seed): sample a subset, build chunks with each strategy x chunk size (plus the
no-chunking baseline), embed them with BGE-small, retrieve, collapse chunk hits to documents (best chunk
score per document) and score Recall@1..20, MRR@10 and NDCG@10. BM25 on whole docs is a lexical reference.

Writes to results/: results.csv (one row per dataset x seed x configuration) and run_info.json
"""
import gc
import json
import platform
import random
import sys
import time

import numpy as np
import pandas as pd

from . import config as C
from . import data as D
from .chunkers import Chunker
from .embed import billed
from .metrics import evaluate
from .retrieval import bm25_tok, doc_scores_maxp, top_k

CONFIGS = [(C.BASELINE, None)] + [(s, c) for s in C.STRATEGIES for c in C.CHUNK_SIZES]


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s)
        torch.cuda.manual_seed_all(s)
    except ImportError:
        pass


def _versions():
    out = {"python": platform.python_version()}
    for mod in ["torch", "sentence_transformers", "transformers", "datasets", "rank_bm25", "numpy", "pandas"]:
        try:
            out[mod] = __import__(mod).__version__
        except Exception:
            pass
    return out


def run_dataset_seed(name, seed, sub, emb, chunker):
    from rank_bm25 import BM25Okapi
    rows = []
    n_docs, nq = len(sub["docs"]), len(sub["qids"])
    doc_ids = np.array(sub["doc_ids"])
    base = dict(dataset=name, seed=seed, n_queries=nq, n_docs=n_docs)

    t0 = time.perf_counter()
    Q = emb.embed(sub["queries"], is_query=True)
    q_time = time.perf_counter() - t0
    q_tok = billed(emb.n_tokens([C.QUERY_PREFIX + q for q in sub["queries"]]))

    # BM25 reference (lexical, whole docs)
    t0 = time.perf_counter()
    bm = BM25Okapi([bm25_tok(d) for d in sub["docs"]])
    t_idx = time.perf_counter() - t0
    t0 = time.perf_counter()
    Dm = np.vstack([bm.get_scores(bm25_tok(q)) for q in sub["queries"]])
    t_s = time.perf_counter() - t0
    m = evaluate(doc_ids[top_k(Dm)].tolist(), sub["qids"], sub["qrels"])
    rows.append({**base, "strategy": "bm25", "chunk_size": "full doc", "n_chunks": n_docs,
                 "avg_chunk_tokens": np.nan, "embedded_tokens": 0, "chunk_time_s": 0.0, "embed_time_s": 0.0,
                 "index_time_s": t_idx, "search_ms_per_query": 1000 * t_s / nq, "query_tokens": 0, **m})

    sent_cache = chunker.prepare_sentences(sub["docs"]) if "semantic" in C.STRATEGIES else None
    for strat, size in CONFIGS:
        t0 = time.perf_counter()
        chunks, owner = chunker.chunk_corpus(sub["docs"], strat, size, sent_cache)
        t_chunk = time.perf_counter() - t0 + (sent_cache["time"] if strat == "semantic" else 0.0)
        lens = emb.n_tokens(chunks)
        t0 = time.perf_counter()
        CE = emb.embed(chunks)
        t_emb = time.perf_counter() - t0
        t0 = time.perf_counter()
        idx = top_k(doc_scores_maxp(Q, CE, owner, n_docs))
        t_s = time.perf_counter() - t0
        m = evaluate(doc_ids[idx].tolist(), sub["qids"], sub["qrels"])
        rows.append({**base, "strategy": strat, "chunk_size": "full doc" if size is None else str(size),
                     "n_chunks": len(chunks),
                     "avg_chunk_tokens": float(np.mean(np.minimum(lens, C.MAX_SEQ_LEN - 2))),
                     "embedded_tokens": billed(lens) + (sent_cache["tokens"] if strat == "semantic" else 0),
                     "chunk_time_s": t_chunk, "embed_time_s": t_emb, "index_time_s": t_chunk + t_emb,
                     "search_ms_per_query": 1000 * (t_s + q_time) / nq, "query_tokens": q_tok, **m})
        print(f"{name:8s} seed={seed:<5d} {strat:9s} {str(size or 'full'):>4s}  chunks={len(chunks):6,d}  "
              f"R@5={m['R@5']:.4f}  R@10={m['R@10']:.4f}  MRR@10={m['MRR@10']:.4f}  "
              f"NDCG@10={m['NDCG@10']:.4f}  index={t_chunk + t_emb:6.1f}s", flush=True)
    return rows


def main(emb=None, data=None):
    """Returns (results DataFrame, embedder, data) so a notebook can reuse them for the frontend."""
    t_start = time.perf_counter()
    if C.SMOKE:
        print("*** P1_SMOKE=1: tiny test run. Do NOT report these numbers. ***")
    set_seed(C.SEEDS[0])
    if emb is None:
        from .embed import Embedder
        emb = Embedder()
    print(f"Embedder: {C.EMBED_MODEL} | device: {getattr(emb, 'device', '?')} ({emb.hardware})")
    if data is None:
        data = D.load_all()
    chunker = Chunker(emb)
    rows = []
    for name in C.DATASETS:
        for seed in C.SEEDS:
            set_seed(seed)
            sub = D.make_subset(data, name, seed)
            rows += run_dataset_seed(name, seed, sub, emb, chunker)
            gc.collect()
    df = pd.DataFrame(rows)
    C.RESULTS.mkdir(parents=True, exist_ok=True)
    df.to_csv(C.RESULTS / "results.csv", index=False)
    total = time.perf_counter() - t_start
    info = {"embed_model": C.EMBED_MODEL, "device": getattr(emb, "device", "?"), "gpu": emb.hardware,
            "llm": None, "llm_tokens": 0, "smoke": C.SMOKE, "total_runtime_s": round(total, 1),
            "total_tokens_embedded": int(df.embedded_tokens.sum() + df.query_tokens.sum()),
            **_versions(),
            "config": {k: getattr(C, k) for k in ["DATASETS", "STRATEGIES", "CHUNK_SIZES", "CHUNK_OVERLAP",
                                                  "SEMANTIC_PERCENTILE", "BASELINE", "SEEDS", "N_QUERIES",
                                                  "CORPUS_SIZE", "K_MAX", "MAX_SEQ_LEN"]},
            "command": " ".join(sys.argv)}
    (C.RESULTS / "run_info.json").write_text(json.dumps(info, indent=2))
    print(f"\nDone: {len(df)} rows -> {C.RESULTS / 'results.csv'} | total wall time {total / 60:.1f} min")
    return df, emb, data


if __name__ == "__main__":
    main()

In [ ]:
%%writefile p1/report.py
"""Build the report tables and figures from results/results.csv:  python -m p1.report

Outputs in results/: T1_quality.csv, T2_cost.csv, tables.md, F1_ndcg_by_strategy.png,
F2_recall_curves.png, findings.txt, cost_summary.json
"""
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from . import config as C

LABEL = {"bm25": "BM25 (full doc, lexical ref.)", "fulldoc": "★ BASELINE: dense, no chunking",
         "fixed": "Fixed-length", "recursive": "Recursive", "semantic": "Semantic"}
ORDER = ["fulldoc", "bm25"] + C.STRATEGIES
METRICS = [("R@5", "Recall@5"), ("R@10", "Recall@10"), ("MRR@10", "MRR@10"), ("NDCG@10", "NDCG@10")]
SIZE_COLORS = ["#4C78A8", "#F58518", "#54A24B", "#B279A2"]
STRAT_COLORS = {"fixed": "#4C78A8", "recursive": "#F58518", "semantic": "#54A24B", "fulldoc": "#222222", "bm25": "#888888"}


def pm(x, fmt="{:.3f}"):
    x = np.asarray(x, dtype=float)
    if len(x) < 2:
        return fmt.format(x.mean())
    return f"{fmt.format(x.mean())} ± {fmt.format(x.std(ddof=1))}"


def _sorted(df):
    df = df.copy()
    df["_o"] = df["strategy"].map({s: i for i, s in enumerate(ORDER)})
    df["_c"] = df["chunk_size"].map(lambda c: 0 if c == "full doc" else int(c))
    return df.sort_values(["dataset", "_o", "_c"]).drop(columns=["_o", "_c"])


def load():
    df = pd.read_csv(C.RESULTS / "results.csv", dtype={"chunk_size": str})
    bl = df[df.strategy == C.BASELINE].set_index(["dataset", "seed"])["NDCG@10"]
    df["dNDCG_vs_base"] = df["NDCG@10"].values - bl.loc[list(zip(df.dataset, df.seed))].values
    return df


def make_t1(df):
    rows = []
    for (ds, st, cs), d in df.groupby(["dataset", "strategy", "chunk_size"], sort=False):
        r = {"dataset": ds, "strategy": st, "chunk_size": cs, "Dataset": ds, "Strategy": LABEL[st], "Chunk size (tok)": cs}
        for k, lab in METRICS:
            r[lab] = pm(d[k])
        r["ΔNDCG@10 vs baseline (paired)"] = "—" if st == C.BASELINE else pm(d["dNDCG_vs_base"], "{:+.3f}").replace("± +", "± ")
        rows.append(r)
    return _sorted(pd.DataFrame(rows)).drop(columns=["dataset", "strategy", "chunk_size"]).reset_index(drop=True)


def make_t2(df):
    rows = []
    for (ds, st, cs), d in df.groupby(["dataset", "strategy", "chunk_size"], sort=False):
        rows.append({"dataset": ds, "strategy": st, "chunk_size": cs, "Dataset": ds, "Strategy": LABEL[st],
                     "Chunk size (tok)": cs, "# chunks": pm(d.n_chunks, "{:,.0f}"),
                     "Avg tokens / chunk": "—" if d.avg_chunk_tokens.isna().all() else f"{d.avg_chunk_tokens.mean():.1f}",
                     "Tokens embedded (index)": f"{d.embedded_tokens.mean():,.0f}",
                     "Chunking time (s)": pm(d.chunk_time_s), "Embedding time (s)": pm(d.embed_time_s),
                     "Indexing time (s)": pm(d.index_time_s), "Search latency (ms/query)": pm(d.search_ms_per_query)})
    return _sorted(pd.DataFrame(rows)).drop(columns=["dataset", "strategy", "chunk_size"]).reset_index(drop=True)


def _agg(df):
    cols = list(dict.fromkeys([k for k, _ in METRICS] + [f"R@{k}" for k in range(1, C.K_MAX + 1)]))
    return df.groupby(["dataset", "strategy", "chunk_size"])[cols].agg(["mean", "std"]).sort_index()


def _ms(agg, ds, st, cs, col):
    row = agg.loc[(ds, st, cs)]
    s = row[(col, "std")]
    return float(row[(col, "mean")]), float(0.0 if np.isnan(s) else s)


def make_f1(df, agg, path):
    datasets = list(dict.fromkeys(df.dataset))
    fig, axes = plt.subplots(1, len(datasets), figsize=(6.2 * len(datasets), 4.6), squeeze=False)
    for ax, ds in zip(axes[0], datasets):
        x = np.arange(len(C.STRATEGIES))
        w = 0.8 / len(C.CHUNK_SIZES)
        tops = []
        for j, cs in enumerate(C.CHUNK_SIZES):
            ms = [_ms(agg, ds, st, str(cs), "NDCG@10") for st in C.STRATEGIES]
            bars = ax.bar(x + (j - (len(C.CHUNK_SIZES) - 1) / 2) * w, [m for m, _ in ms], w,
                          yerr=[s for _, s in ms], capsize=4, color=SIZE_COLORS[j], label=f"chunk size {cs} tokens")
            for b, (m, sd) in zip(bars, ms):
                ax.text(b.get_x() + b.get_width() / 2, m + sd + 0.01, f"{m:.3f}", ha="center", va="bottom", fontsize=8)
                tops.append(m + sd)
        bm, bs = _ms(agg, ds, C.BASELINE, "full doc", "NDCG@10")
        ax.axhline(bm, color="black", ls="--", lw=1.4, label=f"★ baseline: no chunking ({bm:.3f})")
        ax.axhspan(bm - bs, bm + bs, color="black", alpha=0.08)
        m25, _ = _ms(agg, ds, "bm25", "full doc", "NDCG@10")
        ax.axhline(m25, color="#888888", ls=":", lw=1.4, label=f"BM25 full doc ({m25:.3f})")
        ax.set_xticks(x, [LABEL[s] for s in C.STRATEGIES])
        ax.set_xlabel("Chunking strategy")
        ax.set_ylabel("NDCG@10 (mean ± std over seeds)")
        ax.set_title(ds.upper())
        ax.set_ylim(0, min(1.0, max(tops + [m25, bm + bs]) * 1.35))
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(axis="y", alpha=0.3)
    fig.suptitle("F1: NDCG@10 by chunking strategy and chunk size", fontsize=12)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def make_f2(df, agg, path):
    datasets = list(dict.fromkeys(df.dataset))
    ls_map = {C.CHUNK_SIZES[0]: "--", C.CHUNK_SIZES[-1]: "-"}
    Ks = np.arange(1, C.K_MAX + 1)
    fig, axes = plt.subplots(1, len(datasets), figsize=(6.4 * len(datasets), 4.8), squeeze=False)
    for ax, ds in zip(axes[0], datasets):
        curves = [(C.BASELINE, "full doc"), ("bm25", "full doc")] + [(s, str(c)) for s in C.STRATEGIES for c in C.CHUNK_SIZES]
        for st, cs in curves:
            vals = [_ms(agg, ds, st, cs, f"R@{k}") for k in Ks]
            m = np.array([v[0] for v in vals])
            s = np.array([v[1] for v in vals])
            lab = LABEL[st] if cs == "full doc" else f"{LABEL[st]} · {cs} tok"
            ls = ls_map.get(int(cs), "-") if cs != "full doc" else ("-" if st == C.BASELINE else ":")
            ax.plot(Ks, m, ls=ls, lw=2.2 if st == C.BASELINE else 1.5, color=STRAT_COLORS[st], label=lab,
                    marker="o" if st == C.BASELINE else None, ms=3)
            ax.fill_between(Ks, m - s, m + s, color=STRAT_COLORS[st], alpha=0.08)
        ax.set_xlabel("K (number of retrieved documents)")
        ax.set_ylabel("Recall@K (fraction of gold docs retrieved)")
        ax.set_title(ds.upper())
        ax.set_xticks([1, 5, 10, 15, 20])
        ax.grid(alpha=0.3)
        ax.set_ylim(0, 1)
        ax.legend(fontsize=7.5, loc="lower right")
    fig.suptitle(f"F2: Recall@K for each strategy (dashed = {C.CHUNK_SIZES[0]} tok, solid = {C.CHUNK_SIZES[-1]} tok)", fontsize=12)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def findings(df):
    out = []
    for ds in dict.fromkeys(df.dataset):
        d = df[df.dataset == ds]
        agg = d.groupby(["strategy", "chunk_size"])[["NDCG@10", "dNDCG_vs_base"]].agg(["mean", "std"]).fillna(0.0)
        chunked = agg.drop(index=[i for i in agg.index if i[0] in (C.BASELINE, "bm25")])
        best = chunked[("NDCG@10", "mean")].idxmax()
        worst = chunked[("NDCG@10", "mean")].idxmin()
        base = agg.loc[(C.BASELINE, "full doc"), ("NDCG@10", "mean")]
        dm, dsd = chunked.loc[best, ("dNDCG_vs_base", "mean")], chunked.loc[best, ("dNDCG_vs_base", "std")]
        verdict = "beats" if dm - dsd > 0 else "is below" if dm + dsd < 0 else "is within seed noise of"
        out.append(f"{ds.upper()}: best chunked config = {LABEL[best[0]]} @ {best[1]} tok (NDCG@10 "
                   f"{chunked.loc[best, ('NDCG@10', 'mean')]:.3f}); it {verdict} the no-chunking baseline "
                   f"({base:.3f}; paired Δ = {dm:+.3f} ± {dsd:.3f}). Worst = {LABEL[worst[0]]} @ {worst[1]} tok "
                   f"({chunked.loc[worst, ('NDCG@10', 'mean')]:.3f}).")
        for st in C.STRATEGIES:
            a, b = (agg.loc[(st, str(c)), ("NDCG@10", "mean")] for c in C.CHUNK_SIZES)
            out.append(f"   {LABEL[st]}: {C.CHUNK_SIZES[0]} → {C.CHUNK_SIZES[-1]} tok changes NDCG@10 by {b - a:+.3f}.")
        nb = int((chunked[("dNDCG_vs_base", "mean")] < 0).sum())
        if nb:
            out.append(f"   Negative result: {nb}/{len(chunked)} chunked configs score below the no-chunking baseline on {ds}.")
    return out


def main():
    df = load()
    agg = _agg(df)
    info = json.loads((C.RESULTS / "run_info.json").read_text()) if (C.RESULTS / "run_info.json").exists() else {}
    t1, t2 = make_t1(df), make_t2(df)
    t1.to_csv(C.RESULTS / "T1_quality.csv", index=False)
    t2.to_csv(C.RESULTS / "T2_cost.csv", index=False)
    make_f1(df, agg, C.RESULTS / "F1_ndcg_by_strategy.png")
    make_f2(df, agg, C.RESULTS / "F2_recall_curves.png")
    fnd = findings(df)
    (C.RESULTS / "findings.txt").write_text("\n".join(fnd) + "\n")
    cost = {"hardware": info.get("gpu", "?"), "embedder": C.EMBED_MODEL,
            "total_runtime_min": round(info.get("total_runtime_s", float("nan")) / 60, 2),
            "total_tokens_embedded": int(df.embedded_tokens.sum() + df.query_tokens.sum()),
            "llm_tokens": 0, "seeds": sorted(df.seed.unique().tolist()),
            "n_queries_per_seed": int(df.n_queries.iloc[0]), "corpus_size_per_seed": int(df.n_docs.iloc[0])}
    (C.RESULTS / "cost_summary.json").write_text(json.dumps(cost, indent=2))
    smoke = "\n> **SMOKE RUN — not reportable.**\n" if info.get("smoke") else ""
    md = (f"# P1 Primary results (SciFact, FiQA){smoke}\n\n"
          f"## T1: retrieval quality (mean ± std over seeds {cost['seeds']}; ★ = baseline; MRR is MRR@10)\n\n"
          + t1.to_markdown(index=False)
          + f"\n\n## T2: chunks and indexing cost (hardware: {cost['hardware']})\n"
          "Semantic indexing time includes sentence embedding; search latency includes query embedding.\n\n"
          + t2.to_markdown(index=False)
          + f"\n\n## Cost\n\nTotal runtime {cost['total_runtime_min']} min · {cost['total_tokens_embedded']:,} tokens "
          "embedded · 0 LLM tokens\n\n## Findings (computed from results.csv)\n\n```\n" + "\n".join(fnd) + "\n```\n")
    (C.RESULTS / "tables.md").write_text(md)
    print(md)
    return t1, t2, fnd


if __name__ == "__main__":
    main()

In [ ]:
%%writefile p1/app.py
"""Gradio frontend:  python -m p1.app [--share]

Tabs: Results (T1/T2/F1/F2 + findings from results/), Live search (compare strategies on any query,
gold docs marked), Chunk inspector (where each strategy cuts a document), Method.
Live indexes are built on demand for the first seed's subset and cached in memory.
"""
import argparse
import html
import json
import os

import numpy as np
import pandas as pd

from . import config as C
from . import data as D
from .chunkers import Chunker
from .report import LABEL

CSS = """
.hero{padding:18px 22px;border-radius:14px;background:linear-gradient(120deg,#1e1b4b,#3730a3 60%,#4f46e5);color:#fff;margin-bottom:6px}
.hero h1{margin:0 0 4px;font-size:1.55rem;color:#fff}.hero p{margin:0;opacity:.85}
.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:10px;margin:10px 0}
.card{border:1px solid rgba(127,127,127,.25);border-radius:12px;padding:10px 14px}
.card .k{font-size:.75rem;text-transform:uppercase;letter-spacing:.04em;opacity:.65}.card .v{font-size:1.2rem;font-weight:650}
.hit{border:1px solid rgba(127,127,127,.25);border-radius:12px;padding:10px 14px;margin-bottom:8px}
.hit.gold{border-color:#16a34a;box-shadow:0 0 0 1px #16a34a inset}
.hit .meta{font-size:.8rem;opacity:.7;margin-bottom:4px}.hit mark{background:#fde68a;color:#111;border-radius:3px;padding:0 2px}
.col-h{font-weight:650;margin:4px 0 6px}
.chunk{display:inline;padding:1px 2px;border-radius:3px;line-height:1.75;color:#111}
.cbadge{display:inline-block;font-size:.65rem;font-weight:700;background:#111;color:#fff;border-radius:4px;padding:0 4px;margin:0 3px}
.grid3{display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:12px}
.panel{border:1px solid rgba(127,127,127,.25);border-radius:12px;padding:10px 12px;font-size:.86rem}
"""
PALETTE = ["#c7d2fe", "#fde68a", "#bbf7d0", "#fecaca", "#e9d5ff", "#bae6fd"]
STRAT_CHOICES = {"★ Baseline: no chunking": "fulldoc", "Fixed-length": "fixed", "Recursive": "recursive", "Semantic": "semantic"}


class LiveIndex:
    """Seed-0 subsets and lazily built chunk indexes for the frontend."""

    def __init__(self, emb, data):
        self.emb, self.chunker = emb, Chunker(emb)
        self.subs = {ds: D.make_subset(data, ds, C.SEEDS[0]) for ds in C.DATASETS}
        self.cache, self.sent = {}, {}

    def index(self, ds, strat, size):
        key = (ds, strat, size)
        if key not in self.cache:
            docs = self.subs[ds]["docs"]
            if strat == "semantic" and ds not in self.sent:
                self.sent[ds] = self.chunker.prepare_sentences(docs)
            chunks, owner = self.chunker.chunk_corpus(docs, strat, size, self.sent.get(ds))
            self.cache[key] = dict(chunks=chunks, C=self.emb.embed(chunks).astype(np.float16),
                                   starts=np.searchsorted(owner, np.arange(len(docs))))
        return self.cache[key]

    def doc_chunks(self, ds, di, strat, size):
        doc = self.subs[ds]["docs"][di]
        if strat == "semantic":
            sc = self.chunker.prepare_sentences([doc])
            return self.chunker.semantic(sc["sents"][0], sc["embs"][0], sc["lens"][0], size)
        return self.chunker.chunk_corpus([doc], strat, size)[0]


def _highlight(doc, chunk):
    i = doc.find(chunk[:60]) if chunk else -1
    if i < 0:
        return html.escape(chunk)
    j = i + len(chunk)
    s, e = max(0, i - 160), min(len(doc), j + 160)
    return (("…" if s else "") + html.escape(doc[s:i]) + "<mark>" + html.escape(doc[i:j]) + "</mark>"
            + html.escape(doc[j:e]) + ("…" if e < len(doc) else ""))


def build_demo(emb=None, data=None):
    import gradio as gr
    if emb is None:
        from .embed import Embedder
        emb = Embedder()
    if data is None:
        data = D.load_all()
    live = LiveIndex(emb, data)

    R = C.RESULTS
    T1 = pd.read_csv(R / "T1_quality.csv", dtype=str) if (R / "T1_quality.csv").exists() else None
    T2 = pd.read_csv(R / "T2_cost.csv", dtype=str) if (R / "T2_cost.csv").exists() else None
    COST = json.loads((R / "cost_summary.json").read_text()) if (R / "cost_summary.json").exists() else {}
    FND = (R / "findings.txt").read_text().splitlines() if (R / "findings.txt").exists() else []
    RES = pd.read_csv(R / "results.csv", dtype={"chunk_size": str}) if (R / "results.csv").exists() else None

    def cards():
        items = [("Hardware", COST.get("hardware", emb.hardware)), ("Total runtime", f"{COST.get('total_runtime_min', '—')} min"),
                 ("Tokens embedded", f"{COST['total_tokens_embedded']:,}" if COST else "—"), ("LLM tokens", "0")]
        if RES is not None:
            for ds in C.DATASETS:
                d = RES[(RES.dataset == ds) & ~RES.strategy.isin([C.BASELINE, "bm25"])]
                if len(d):
                    a = d.groupby(["strategy", "chunk_size"])["NDCG@10"].mean()
                    i = a.idxmax()
                    items.append((f"Best on {ds}", f"{LABEL[i[0]]} · {i[1]} — {a.max():.3f}"))
        return '<div class="cards">' + "".join(
            f'<div class="card"><div class="k">{k}</div><div class="v">{html.escape(str(v))}</div></div>' for k, v in items) + "</div>"

    def filter_tables(ds):
        f = lambda T: T if (T is None or ds == "All") else T[T.Dataset == ds]
        return f(T1), f(T2)

    def example_choices(ds):
        s = live.subs[ds]
        return [f"{q} │ {t[:110]}" for q, t in zip(s["qids"][:80], s["queries"][:80])]

    def pick_example(ds, choice):
        if not choice:
            return gr.update()
        s = live.subs[ds]
        return s["queries"][s["qids"].index(choice.split(" │ ")[0])]

    def gold_for(ds, query):
        s = live.subs[ds]
        for q, t in zip(s["qids"], s["queries"]):
            if t.strip() == query.strip():
                return set(s["qrels"][q])
        return set()

    def live_search(ds, strategies, size, query, k):
        if not query or not query.strip():
            return "<p>Type a query or pick a benchmark query.</p>"
        if not strategies:
            return "<p>Select at least one strategy.</p>"
        s = live.subs[ds]
        gold, k = gold_for(ds, query), int(k)
        qv = emb.embed([query], is_query=True)[0]
        cols = []
        for lab in strategies:
            st = STRAT_CHOICES[lab]
            sz = None if st == C.BASELINE else int(size)
            L = live.index(ds, st, sz)
            sc = L["C"].astype(np.float32) @ qv
            Dm = np.maximum.reduceat(sc, L["starts"])
            top = np.argsort(-Dm, kind="stable")[:k]
            n_gold = sum(s["doc_ids"][d] in gold for d in top)
            head = lab + ("" if sz is None else f" · {sz} tok") + (f" — {n_gold}/{len(gold)} gold in top-{k}" if gold else "")
            body = ""
            for r, di in enumerate(top, 1):
                lo = L["starts"][di]
                hi = L["starts"][di + 1] if di + 1 < len(L["starts"]) else len(sc)
                ch = L["chunks"][lo + int(np.argmax(sc[lo:hi]))]
                did = s["doc_ids"][di]
                g = did in gold
                body += (f'<div class="hit {"gold" if g else ""}"><div class="meta">#{r} · doc {html.escape(did)} · '
                         f'score {Dm[di]:.3f}{" · ✅ gold" if g else ""}</div>{_highlight(s["docs"][di], ch)}</div>')
            cols.append(f'<div><div class="col-h">{html.escape(head)}</div>{body}</div>')
        note = "" if gold else "<p style='opacity:.7'>Free-text query: no gold labels, so hits are not marked.</p>"
        return note + '<div class="grid3">' + "".join(cols) + "</div>"

    def doc_choices(ds):
        s = live.subs[ds]
        gold = sorted(set(d for q in s["qids"] for d in s["qrels"][q]))
        idx = {d: i for i, d in enumerate(s["doc_ids"])}
        return [f"{d} │ {s['docs'][idx[d]][:90].replace(chr(10), ' ')}" for d in gold[:150]]

    def inspect_doc(ds, choice):
        if not choice:
            return ""
        s = live.subs[ds]
        di = s["doc_ids"].index(choice.split(" │ ")[0])
        panels = []
        for st in C.STRATEGIES:
            for cs in C.CHUNK_SIZES:
                chs = live.doc_chunks(ds, di, st, cs)
                lens = emb.n_tokens(chs)
                spans = "".join(f'<span class="cbadge">{i + 1}·{l}t</span><span class="chunk" style="background:'
                                f'{PALETTE[i % len(PALETTE)]}">{html.escape(c)}</span> ' for i, (c, l) in enumerate(zip(chs, lens)))
                panels.append(f'<div class="panel"><div class="col-h">{LABEL[st]} · {cs} tok — {len(chs)} chunk(s)</div>{spans}</div>')
        n = emb.n_tokens(s["docs"][di])[0]
        return f"<p><b>Doc {html.escape(s['doc_ids'][di])}</b> · {n} tokens</p>" + '<div class="grid3">' + "".join(panels) + "</div>"

    method_md = f"""
**Datasets:** {", ".join(d.upper() for d in C.DATASETS)} (BEIR test split). Per seed: {C.N_QUERIES} queries, {C.CORPUS_SIZE} docs (all gold + random distractors). Seeds {C.SEEDS}.
**Embedder:** `{C.EMBED_MODEL}` (fp16 on GPU), BGE query instruction, cosine similarity. **No LLM** (retrieval-only project).
**Strategies** (budget in embedder tokens, overlap {C.CHUNK_OVERLAP}):
- *Fixed-length* — consecutive windows of exactly N tokens.
- *Recursive* — split on paragraph → line → sentence → word, greedily packed up to N tokens.
- *Semantic* — sentence embeddings; new chunk where adjacent-sentence distance exceeds the doc's {C.SEMANTIC_PERCENTILE}th percentile; N is a hard cap.
- ★ *Baseline* — whole document embedded as one vector (truncated at {C.MAX_SEQ_LEN} tokens). BM25 on whole docs is a lexical reference.
**Doc ranking:** MaxP (doc score = best chunk score). **Metrics:** Recall@5/10, MRR@10, NDCG@10, Recall@1–20; mean ± std over seeds.
"""
    theme = gr.themes.Soft(primary_hue="indigo", secondary_hue="amber", neutral_hue="slate",
                           font=[gr.themes.GoogleFont("Inter"), "system-ui", "sans-serif"])
    gr6 = int(gr.__version__.split(".")[0]) >= 6           # Gradio 6 moved theme/css from Blocks() to launch()
    blocks_kw = {} if gr6 else dict(theme=theme, css=CSS)
    with gr.Blocks(title="P1 · Chunking Benchmark", **blocks_kw) as demo:
        gr.HTML('<div class="hero"><h1>Chunking Benchmark · SciFact & FiQA</h1>'
                '<p>Fixed-length vs recursive vs semantic chunking at 128 / 256 tokens · BGE-small · 3 seeds</p></div>')
        gr.HTML(cards())
        with gr.Tab("Results"):
            if T1 is None:
                gr.Markdown("No results yet: run `python -m p1.run_experiment` and `python -m p1.report` first.")
            else:
                ds_f = gr.Radio(["All"] + C.DATASETS, value="All", label="Dataset")
                gr.Markdown("**T1 · Retrieval quality** (mean ± std over seeds; ★ = baseline; Δ is paired per seed)")
                t1_out = gr.Dataframe(value=T1, wrap=True, interactive=False)
                gr.Markdown("**T2 · Number of chunks and indexing cost**")
                t2_out = gr.Dataframe(value=T2, wrap=True, interactive=False)
                ds_f.change(filter_tables, ds_f, [t1_out, t2_out])
                for f, lab in [("F1_ndcg_by_strategy.png", "F1 · NDCG@10 by strategy and chunk size"),
                               ("F2_recall_curves.png", "F2 · Recall@K, K = 1–20")]:
                    if (R / f).exists():
                        gr.Image(str(R / f), label=lab)
                if FND:
                    gr.Markdown("**Findings (computed from results.csv):**\n\n" + "\n".join(f"- {x.strip()}" for x in FND))
        with gr.Tab("Live search"):
            with gr.Row():
                ds_s = gr.Dropdown(C.DATASETS, value=C.DATASETS[0], label="Dataset")
                size_s = gr.Radio([str(c) for c in C.CHUNK_SIZES], value=str(C.CHUNK_SIZES[0]), label="Chunk size (tokens)")
                k_s = gr.Slider(1, 10, value=5, step=1, label="Top-k docs")
            strat_s = gr.CheckboxGroup(list(STRAT_CHOICES), value=["Fixed-length", "Recursive", "Semantic"], label="Strategies to compare")
            ex_s = gr.Dropdown(example_choices(C.DATASETS[0]), label="Benchmark query (gold docs get ✅)", value=None)
            q_s = gr.Textbox(label="Query", placeholder="e.g. Does vitamin D supplementation reduce fracture risk?")
            go = gr.Button("Search", variant="primary")
            gr.Markdown("<small>The first search for a dataset / strategy / size builds its index (a few seconds on GPU).</small>")
            res_s = gr.HTML()
            ds_s.change(lambda d: gr.update(choices=example_choices(d), value=None), ds_s, ex_s)
            ex_s.change(pick_example, [ds_s, ex_s], q_s)
            go.click(live_search, [ds_s, strat_s, size_s, q_s, k_s], res_s)
            q_s.submit(live_search, [ds_s, strat_s, size_s, q_s, k_s], res_s)
        with gr.Tab("Chunk inspector"):
            ds_c = gr.Dropdown(C.DATASETS, value=C.DATASETS[0], label="Dataset")
            doc_c = gr.Dropdown(doc_choices(C.DATASETS[0]), label=f"Document (gold docs of the seed-{C.SEEDS[0]} subset)", value=None)
            out_c = gr.HTML()
            ds_c.change(lambda d: gr.update(choices=doc_choices(d), value=None), ds_c, doc_c)
            doc_c.change(inspect_doc, [ds_c, doc_c], out_c)
        with gr.Tab("Method"):
            gr.Markdown(method_md)

    demo._p1_launch_kw = dict(theme=theme, css=CSS) if gr6 else {}
    demo._p1_fns = dict(live_search=live_search, inspect_doc=inspect_doc, doc_choices=doc_choices,
                        example_choices=example_choices, pick_example=pick_example, filter_tables=filter_tables)
    return demo


def launch(demo, **kw):
    return demo.launch(**{**demo._p1_launch_kw, **kw})


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--share", action="store_true", help="create a public gradio.live link")
    ap.add_argument("--port", type=int, default=int(os.environ.get("PORT", 7860)))
    a = ap.parse_args()
    launch(build_demo(), share=a.share, server_port=a.port)

## 3. Self-test the metrics, then run the full experiment

In [ ]:
!python -m p1.metrics

Main step. Runs in this kernel (so the frontend can reuse the loaded model and data) and prints one line per
configuration: chunks, Recall@5/10, MRR@10, NDCG@10 and indexing time, for 2 datasets x 3 seeds.

In [ ]:
from p1 import run_experiment
results, EMB, DATA = run_experiment.main()

## 4. Tables T1, T2 and figures F1, F2

In [ ]:
from p1 import report
from IPython.display import Image, display
T1, T2, FINDINGS = report.main()
display(Image("results/F1_ndcg_by_strategy.png"))
display(Image("results/F2_recall_curves.png"))

## 5. Download the results folder
Unzip it into `27_P1_Primary/results/` in the repository and commit.

In [ ]:
import shutil
shutil.make_archive("results", "zip", "results")
try:
    from google.colab import files
    files.download("results.zip")
except Exception:
    print("Saved results.zip")

## 6. Frontend
Starts the Gradio app with four tabs: **Results** (T1/T2/F1/F2 + findings), **Live search** (compare strategies
on any query; gold docs marked ✅), **Chunk inspector** (where each strategy cuts a document) and **Method**.
Click the `gradio.live` link it prints. Stop the cell to shut it down.

In [ ]:
from p1.app import build_demo, launch
demo = build_demo(EMB, DATA)
launch(demo, share=True, debug=True)